Official reproduction copy. Code cells reuse the accepted experiment paths and source receipt. The experiment remains unchanged. This notebook is not a new training authorization.


# R2-U2 Option B ? fresh two-epoch isolated experiment

Use Account B's prepared Tesla T4. First run the UPDATED software-verification notebook: all 17 tests must pass with zero skips. Its previous 10-test report remains historical evidence and cannot authorize changed sources.

Upload only the updated experiments/r2_u2 folder, including software_sources. Existing Drive data and metadata are read-only inputs. Shared Trainer, R3, historical failed outputs and official R2 locations are never written. Git-pinned sources are staged in scratch. Only a new private experiment run and report directories receive outputs.

Set PROJECT_ROOT and SOFTWARE_REPORT, then deliberately set AUTHORIZE_FRESH_EXPERIMENT=True. Run cells in order. This notebook never resumes a checkpoint. Do not rerun launch to recover an interruption: preserve outputs and inspect the previous process first.

The prospective option_b_v1 contract is in configs/option_b_policy.json. The exact historical recipe is unchanged. Every attempt is explained by observed optimizer calls and scaler transitions; finite applied norms are separated from overflow events. No arbitrary skip threshold is used. A final skipped attempt is disclosed without claiming stability at the backed-off scale. Passing experiment checks is pending team review, not official acceptance or R3 promotion.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, uuid, zipfile, traceback

PROJECT_ROOT = '/content/drive/MyDrive/Colab_R2_U1_Workspace'  # Change only to your actual project location.
ROOT = Path(PROJECT_ROOT).resolve()
EXP = ROOT / 'experiments/r2_u2'
assert EXP.is_dir(), f'Upload experiments/r2_u2 into {ROOT}'
TOKEN = uuid.uuid4().hex
SCRATCH = Path('/content') / ('echoproof-r2-u2-software-' + TOKEN)
assert not SCRATCH.exists()
SCRATCH.mkdir()
(SCRATCH / 'tmp').mkdir()
OUT = EXP / 'reports' / ('launch_' + TOKEN)
OUT.mkdir(parents=True, exist_ok=False)
REPORT = {'task': 'R2-U2 Option B launch preparation', 'started_utc': datetime.now(timezone.utc).isoformat(),
          'research_accepted': False, 'software_gate_passed': False, 'status': 'started',
          'drive_project': str(ROOT), 'scratch': str(SCRATCH)}
def save_report():
    (OUT / 'verification.json').write_text(json.dumps(REPORT, indent=2, sort_keys=True, allow_nan=False))
def sha_bytes(b):
    return hashlib.sha256(b).hexdigest()
CHECKED = False
print('New verification reports:', OUT)
save_report()

SOFTWARE_REPORT = "/content/drive/MyDrive/Colab_R2_U1_Workspace/experiments/r2_u2/reports/software_70fc2d647510481d93297f01ba76665c/verification.json"
AUTHORIZE_FRESH_EXPERIMENT = True  # Set True deliberately for this separately identified fresh run.


In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
%pip install -q --force-reinstall --no-cache-dir --no-deps transformers==5.22.0

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-c", "import transformers; print(transformers.__version__)"],
    capture_output=True, text=True
)
print("Transformers:", result.stdout.strip())
print("Errors:", result.stderr.strip())

## Verify and stage the reviewed sources

Missing or altered uploads stop launch. Existing Drive shared sources remain untouched.


In [ ]:
CHECKED = False
try:
    manifest_path = EXP / 'colab_source_manifest.json'
    manifest = json.loads(manifest_path.read_text())
    REPORT['manifest_sha256'] = sha_bytes(manifest_path.read_bytes())
    REPORT['expected_source_hashes'] = manifest['files']
    for name, expected in manifest['files'].items():
        rel = Path(name)
        assert not rel.is_absolute() and '..' not in rel.parts, name
        if name.startswith('experiments/r2_u2/'):
            source = ROOT / rel
            assert source.is_file(), f'Missing upload: {name}'
            assert source.resolve().is_relative_to(EXP), f'Unsafe source path: {name}'
            data = source.read_bytes()
            assert sha_bytes(data) == expected, f'Mismatched upload: {name}'
            if not name.startswith('experiments/r2_u2/software_sources/'):
                target = SCRATCH / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open('xb') as f: f.write(data)
        else:
            assert name.startswith('src/'), f'Unreviewed manifest dependency: {name}'
    snapshot = json.loads((EXP / 'software_sources/snapshot.json').read_text())
    assert snapshot['git_commit'] == 'c7195dbfa4ccb506bf058102095df92315948d20', 'Unreviewed source snapshot'
    REPORT['shared_source_snapshot'] = snapshot
    with zipfile.ZipFile(EXP / 'software_sources/shared_sources.zip') as bundle:
        assert len(bundle.namelist()) == len(set(bundle.namelist()))
        assert set(bundle.namelist()) == set(snapshot['files'])
        for name, info in snapshot['files'].items():
            rel = Path(name)
            assert not rel.is_absolute() and '..' not in rel.parts
            data = bundle.read(name)
            assert len(data) == info['size'] and sha_bytes(data) == info['sha256'], name
            blob = data if sha_bytes(data) == info['git_content_sha256'] else data.replace(b'\r\n', b'\n')
            assert sha_bytes(blob) == info['git_content_sha256'], name
            assert hashlib.sha1(b'blob ' + str(len(blob)).encode() + b'\0' + blob).hexdigest() == info['git_blob_sha1'], name
            target = SCRATCH / rel
            assert target.resolve().is_relative_to(SCRATCH)
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open('xb') as f: f.write(data)
    for name, expected in manifest['files'].items():
        if name.startswith('src/'):
            assert sha_bytes((SCRATCH / name).read_bytes()) == expected, f'Snapshot/manifest disagreement: {name}'
    REPORT['existing_drive_shared_source_differences'] = [name for name, expected in manifest['files'].items()
        if name.startswith('src/') and (not (ROOT / name).is_file() or sha_bytes((ROOT / name).read_bytes()) != expected)]
    REPORT['status'] = 'sources_verified'
    CHECKED = True
    print('Verified snapshot; existing Drive shared sources untouched.')
    print('Existing Drive source differences:', REPORT['existing_drive_shared_source_differences'])
except BaseException:
    REPORT.update(status='source_verification_failed', traceback=traceback.format_exc())
    raise
finally:
    save_report()


## Fresh experimental execution

Requires the new 17-test report. Canonical plan shards are hashed and staged to scratch; only B training and selection recordings are decoded. The pinned AST model is downloaded to scratch. The run prints its unique persistent output path. Do not run other experiment writers concurrently.


In [ ]:
assert CHECKED, 'Source verification must pass first'
assert AUTHORIZE_FRESH_EXPERIMENT is True, 'Set the explicit fresh-run launch switch'
assert SOFTWARE_REPORT, 'Set the new 17-test verification.json path'
# Copy only verification metadata needed by the scratch launcher.
for relative in ['colab_source_manifest.json', 'software_sources/snapshot.json', 'software_sources/shared_sources.zip']:
    source = EXP / relative
    target = SCRATCH / 'experiments/r2_u2' / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open('xb') as f: f.write(source.read_bytes())
ENV = dict(os.environ, PYTHONPATH=str(SCRATCH / 'src'), PYTHONDONTWRITEBYTECODE='1',
           CUBLAS_WORKSPACE_CONFIG=':4096:8', TMPDIR=str(SCRATCH / 'tmp'),
           TMP=str(SCRATCH / 'tmp'), TEMP=str(SCRATCH / 'tmp'),
           TORCH_HOME=str(SCRATCH / 'torch'), HF_HOME=str(SCRATCH / 'hf'),
           XDG_CACHE_HOME=str(SCRATCH / 'cache'))
command = [sys.executable, '-B', str(SCRATCH / 'experiments/r2_u2/scripts/launch.py'),
           '--data-root', str(ROOT), '--software-report', SOFTWARE_REPORT,
           '--authorize-fresh-experiment']
REPORT['command'] = command
REPORT['software_report'] = SOFTWARE_REPORT
try:
    # Exclusive log prevents accidentally rerunning this launch cell.
    with (OUT / 'launch.log').open('x') as log:
        process = subprocess.Popen(command, cwd=SCRATCH, env=ENV, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line); log.flush(); os.fsync(log.fileno())
        returncode = process.wait()
    REPORT.update(returncode=returncode, status='experiment_checks_pending_review' if returncode == 0 else 'launch_or_experiment_failed')
except BaseException:
    REPORT.update(status='launch_failed_or_interrupted', traceback=traceback.format_exc())
    # A Colab interruption does not prove the child writer stopped. Never launch
    # another run or delete outputs until its process status is inspected.
    raise
finally:
    REPORT['finished_utc'] = datetime.now(timezone.utc).isoformat()
    save_report()
print('Launch report:', OUT)
print('No checkpoint has been accepted or promoted.')


In [ ]:
import torch
import subprocess
import sys

print("Notebook CUDA:", torch.cuda.is_available())

result = subprocess.run(
    [sys.executable, "-c",
     "import torch; print('CUDA:', torch.cuda.is_available()); "
     "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"],
    capture_output=True, text=True
)

print("Subprocess:", result.stdout)
print("Errors:", result.stderr)

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

## Return the launch report

Download the small log/report ZIP below. The checkpoints and experiment_result.json remain under the printed experiments/r2_u2/runs/option_b_<uuid> directory on Drive. Share the result JSON for independent analysis; do not promote a checkpoint.


In [ ]:
from google.colab import files
RESULT_ZIP = SCRATCH / 'software_verification_results.zip'
with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.iterdir()):
        if path.is_file(): bundle.write(path, path.name)
files.download(str(RESULT_ZIP))
